In [2]:
# =====================================================================
# VERITASGUIDE V7.0 - RESEÑAS EN REDES SOCIALES POR PLATO/CAPITAL (LATAM)
# =====================================================================
# 1. INSTALACIÓN DE DEPENDENCIAS
# !pip install google-genai pydantic -q

import json
import os
import time
import re
from typing import Literal
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

# =====================================================================
# 2. API KEY VÍA VARIABLE DE ENTORNO (nunca hardcodeada)
# =====================================================================
# Antes de ejecutar, configurá la variable de entorno:
#   Linux/Mac:            export GEMINI_API_KEY="tu_clave_aqui"
#   Windows PowerShell:    $env:GEMINI_API_KEY="tu_clave_aqui"
import os
os.environ["GEMINI_API_KEY"] = "YOURAPIKEYHERE"
API_KEY = os.environ.get("GEMINI_API_KEY")
if not API_KEY:
    raise EnvironmentError(
        "⚠️ No se encontró GEMINI_API_KEY en las variables de entorno. "
        "Configurala antes de ejecutar este script (ver comentario arriba)."
    )
client = genai.Client(api_key=API_KEY)

# =====================================================================
# 3. BASE DE DATOS DE LA GUÍA (tomada del informe ya elaborado)
# =====================================================================
# En vez de volver a preguntarle a la IA cuál es el plato de cada capital,
# se usa directamente lo que ya está consolidado en el informe comparativo.
# Podés editar/ampliar esta lista con más países o platos si hace falta.

GUIA_CAPITALES = [
    {"pais": "Argentina", "capital": "Buenos Aires",
     "platos": ["Asado", "Choripán con chimichurri"]},
    {"pais": "Perú", "capital": "Lima",
     "platos": ["Cebiche", "Causa Limeña", "Lomo Saltado"]},
    {"pais": "México", "capital": "Ciudad de México",
     "platos": ["Mole", "Tacos al Pastor", "Chiles en Nogada"]},
    {"pais": "Guatemala", "capital": "Ciudad de Guatemala",
     "platos": ["Pepián", "Kak'ik", "Hilachas"]},
    {"pais": "República Dominicana", "capital": "Santo Domingo",
     "platos": ["La Bandera", "Mangú", "Sancocho"]},
    {"pais": "Panamá", "capital": "Ciudad de Panamá",
     "platos": ["Sancocho de gallina", "Arroz con Pollo", "Carimañolas de yuca"]},
    {"pais": "Costa Rica", "capital": "San José",
     "platos": ["Gallo Pinto", "Casado", "Olla de Carne"]},
    {"pais": "Colombia", "capital": "Bogotá",
     "platos": ["Ajiaco Santafereño", "Tamal Bogotano", "Chocolate Completo"]},
    {"pais": "Brasil", "capital": "Brasilia",
     "platos": ["Feijoada", "Pão de Queijo", "Picanha"]},
    {"pais": "Chile", "capital": "Santiago",
     "platos": ["Pastel de Choclo", "Empanadas de Pino", "Cazuela"]},
]

RedSocial = Literal["Reddit", "X (Twitter)", "Facebook", "TikTok", "Instagram", "Google Maps/TripAdvisor"]

# =====================================================================
# 4. ESQUEMAS DE DATOS
# =====================================================================
class LocalRecomendado(BaseModel):
    nombre_local: str = Field(description="Nombre del restaurante, mercado, puesto o local recomendado.")
    direccion_o_zona: str = Field(description="Ubicación aproximada, barrio o zona del local en la capital.")
    red_social_origen: RedSocial = Field(description="Red social donde se encontró la recomendación con más menciones u opiniones.")
    resena_redes: str = Field(description="Resumen breve de por qué la gente lo recomienda en redes (sazón, precio, tradición, porción).")
    indice_honestidad: int = Field(ge=0, le=100, description="Puntaje 0-100 según autenticidad de las opiniones frente al 'hype' pago/publicitario.")

class ReportePlatoCapital(BaseModel):
    pais: str
    capital: str
    plato: str
    mejores_lugares: list[LocalRecomendado] = Field(
        description="Lista de EXACTAMENTE TRES (3) de los mejores lugares validados en redes para comer este plato."
    )
    resumen_vibe_gastronomica: str = Field(description="Cómo se vive la cultura de este plato en la capital, según lo que dicen las redes.")

# =====================================================================
# 5. CONTROL DE CUOTAS (ANTI-429 / ANTI-503)
# =====================================================================
def llamar_gemini_con_control_cuota(prompt_texto, config_tipo, intentos=5):
    for _ in range(intentos):
        try:
            return client.models.generate_content(
                model='gemini-2.5-flash',
                contents=prompt_texto,
                config=config_tipo
            )
        except Exception as e:
            error_str = str(e)

            if "429" in error_str or "RESOURCE_EXHAUSTED" in error_str:
                if "PerDay" in error_str or "requests_per_day" in error_str.lower():
                    print("\n🚨 ERROR CRÍTICO: se agotó la cuota GRATUITA DIARIA. Reintentá después de medianoche (PT).")
                    raise e
                segundos_match = re.search(r"retry in ([\d\.]+)s", error_str)
                segundos_espera = float(segundos_match.group(1)) + 2.0 if segundos_match else 35.0
                print(f"🛑 Límite temporal (429). Durmiendo {segundos_espera:.2f}s...")
                time.sleep(segundos_espera)

            elif "503" in error_str or "UNAVAILABLE" in error_str:
                print("⚠️ Servidor saturado (503). Esperando 10s...")
                time.sleep(10)
            else:
                raise e

    raise Exception("❌ Intentos máximos agotados debido a restricciones persistentes del servidor.")

# =====================================================================
# 6. BÚSQUEDA DIRIGIDA A REDDIT, X, FACEBOOK, TIKTOK E INSTAGRAM
# =====================================================================
SITIOS_BUSQUEDA = {
    "Reddit": "site:reddit.com",
    "X (Twitter)": "site:x.com OR site:twitter.com",
    "Facebook": "site:facebook.com",
    "TikTok": "site:tiktok.com",
    "Instagram": "site:instagram.com",
}

def construir_prompt_busqueda(pais: str, capital: str, plato: str) -> str:
    operadores = " OR ".join(SITIOS_BUSQUEDA.values())
    return (
        f"Busca en internet, priorizando estas redes sociales ({operadores}), "
        f"recomendaciones reales y recientes de dónde comer '{plato}' en {capital}, {pais}. "
        f"Necesito exactamente 3 lugares distintos (restaurantes, mercados, picadas o puestos callejeros) "
        f"con el mayor respaldo de comentarios genuinos de usuarios, evitando publicidad paga o notas de prensa genéricas. "
        f"Para cada lugar indicá: nombre, zona/dirección aproximada, en qué red social se mencionó con más fuerza, "
        f"qué dicen los usuarios sobre sabor/precio/porciones, y qué tan orgánica/confiable parece la recomendación "
        f"frente al posible 'hype' publicitario."
    )

def buscar_lugares_para_plato(pais: str, capital: str, plato: str) -> ReportePlatoCapital:
    print(f"📱 Buscando recomendaciones para '{plato}' en {capital}, {pais}...")
    prompt = construir_prompt_busqueda(pais, capital, plato)

    config_search = types.GenerateContentConfig(
        temperature=0.3,
        tools=[types.Tool(google_search=types.GoogleSearch())]
    )
    response_search = llamar_gemini_con_control_cuota(prompt, config_search)

    prompt_json = (
        f"A partir de esta investigación sobre dónde comer '{plato}' en {capital}, {pais}, "
        f"extraé y estructurá la información en el JSON pedido, con exactamente 3 locales. "
        f"Investigación: {response_search.text}"
    )
    config_json = types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=ReportePlatoCapital,
        temperature=0.1
    )
    response_json = llamar_gemini_con_control_cuota(prompt_json, config_json)

    if hasattr(response_json, 'parsed') and response_json.parsed:
        return response_json.parsed
    datos = json.loads(response_json.text)
    return ReportePlatoCapital(**datos)

# =====================================================================
# 7. EJECUCIÓN Y EXPORTACIÓN
# =====================================================================
def _imprimir_reporte(reporte: ReportePlatoCapital) -> None:
    print("\n" + "=" * 60)
    print(f"📍 {reporte.capital.upper()} ({reporte.pais.upper()}) — 🍲 {reporte.plato}")
    print(f"✨ {reporte.resumen_vibe_gastronomica}")
    print("=" * 60)
    for idx, local in enumerate(reporte.mejores_lugares, 1):
        print(f"  {idx}. {local.nombre_local} ({local.direccion_o_zona})")
        print(f"     🔹 Red: {local.red_social_origen}")
        print(f"     🔹 Reseña: {local.resena_redes}")
        print(f"     🔹 Índice de honestidad: {local.indice_honestidad}/100")

def ejecutar_guia_completa(
    guia: list[dict] = GUIA_CAPITALES,
    pausa_entre_llamadas: float = 4.0,
    salida_json: str = "resenas_redes_latam.json",
) -> list[dict]:
    """
    Recorre cada capital y cada plato de la guía, busca 3 lugares recomendados
    en redes sociales para cada uno, y guarda todo en un JSON al final.

    Para pruebas rápidas, pasá un slice, ej: ejecutar_guia_completa(GUIA_CAPITALES[:2])
    """
    resultados = []
    for entrada in guia:
        for plato in entrada["platos"]:
            try:
                reporte = buscar_lugares_para_plato(entrada["pais"], entrada["capital"], plato)
                resultados.append(reporte.model_dump())
                _imprimir_reporte(reporte)
            except Exception as e:
                print(f"❌ Error con '{plato}' en {entrada['capital']}: {e}")
            time.sleep(pausa_entre_llamadas)  # margen extra para no chocar límites por minuto

    with open(salida_json, "w", encoding="utf-8") as f:
        json.dump(resultados, f, ensure_ascii=False, indent=2)
    print(f"\n💾 Resultados guardados en {salida_json}")
    return resultados

# =====================================================================
# 8. MAIN
# =====================================================================
if __name__ == "__main__":
    print("=======================================================================")
    print("  VERITASGUIDE V7.0 - RESEÑAS EN REDES SOCIALES POR PLATO/CAPITAL ")
    print("=======================================================================\n")

    try:
        ejecutar_guia_completa()
    except Exception as e:
        print(f"\n❌ Se detuvo la ejecución debido a un error: {str(e)}")

  VERITASGUIDE V7.0 - RESEÑAS EN REDES SOCIALES POR PLATO/CAPITAL 

📱 Buscando recomendaciones para 'Asado' en Buenos Aires, Argentina...

📍 BUENOS AIRES (ARGENTINA) — 🍲 Asado
✨ La cultura del Asado en Buenos Aires es vibrante y diversa, abarcando desde parrillas auténticas y económicas con un fuerte arraigo local, hasta experiencias gourmet sofisticadas que reinterpretan la tradición. Las redes sociales reflejan esta riqueza, destacando tanto la calidad de la carne y el sabor tradicional, como la innovación y el ambiente único de cada lugar, con un fuerte énfasis en la experiencia compartida.
  1. Gran Paraíso (Gral. José Garibaldi 1428, La Boca)
     🔹 Red: Google Maps/TripAdvisor
     🔹 Reseña: Ofrece excelentes cortes a precio de barrio. Recomiendan la parrilla completa, mollejas, choripán crocante, burrata artesanal, empanadas y papas al plomo.
     🔹 Índice de honestidad: 92/100
  2. Rufino (Avenida Presidente Manuel Quintana 465, subsuelo del hotel Mio Buenos Aires, Recoleta)
  

KeyboardInterrupt: 